In [1]:
using LinearAlgebra, BenchmarkTools, PolynomialRoots, StaticArrays, DataStructures

# Fully-split velocity Lagrangian PDMP (Technical WIP - not complete yet)
The main branch has the funcitonal code for the Split Lagrangian PDMP and Covariance Adaptive BPS. This part is under development.

## Mathemagical preliminaries
### A quick outline
Our goal shall be to define a set of exactly solvable problems to see that the fully-split velocity Lagrangian PDMP can be handled almost entirely analytically. First we establish what the rates and dynamics look like. Then we show that the rates are cubic in each velocity component, and that the dynamics are exactly solvable. We shall compute and store the corresponding coefficients of the dynamics ODE and rates as tensors. Then we shall implement a program to compute the rates and update rules for the split-velocity part of the PDMP. This will enable us to merge this code with the previously written PDMP code to finally run this in parallell to CA-BPS or SL-PDMP.


### Splitting equations and general rate solution
Consider, for simplicity, a general split PDMP with each transitions $\alpha \to \beta$, i.e. only the split dynamic changes (Other events are entirely possible to consider in the same framework, but clutter up notation, so let us leave them out for now).  Let $\lambda(\alpha \to \beta)$ be the associated rates. To preserve a distribution $\mu$ the split flow is taken to satisfy

$
-\mu^{-1}\text{div}_\alpha(\mu \Phi^\alpha) = \sum_\beta  \lambda(\alpha \to \beta) - \lambda(\alpha \to \beta)
$

and denoting the divergence (sign?) on the LHS by $A_\alpha$, and $\lambda_{\alpha\beta} =\lambda(\alpha \to \beta) $ we get

$
A_\alpha = \sum_\beta \lambda_{\alpha\beta}-\lambda_{\beta\alpha}
$

Of course, if the original $\Phi$ satisfies $\text{div}(\mu \Phi)=0$ then so does the sum of split flows, as the sum of divergences $A_\alpha$ vanish by linearity of the divergence operator.

In trying to define the rates we have to solve a constraint on an anti-symmetric matrix. Clearly, adding any symmetric part to $\lambda_{\alpha\beta}$ is inconsequential for the constraint above. Effectively, under such a transformation, the altered flow ${\alpha \to \beta}$ is offset by an equivalent flow $\beta \to \alpha$, and in simulation we will get more or fewer events but the overall divergence is not altered. Since events are typically costly we generally will want $\lambda_{{\alpha\beta}}$ to have as small a symmetric part as possible. Alas, $\lambda_{{\alpha\beta}} \geq 0 $ so the symmetric part cannot be zero identically.

Rather than solving the above positivity and anti-symmetry constraints on $\lambda_{{\alpha\beta}}$ we can assume 

$
\lambda_{{\alpha\beta}} = [\rho_{\alpha\beta}]^+
$

where $\rho_{\alpha\beta}$ is some anti-symmetric matrix. Then 

$
\lambda_{\alpha\beta} - \lambda_{\beta\alpha}= [\rho_{\alpha\beta}]^+ - [\rho_{\beta\alpha}]^+ = [\rho_{\alpha\beta}]^+-[-\rho_{\alpha\beta}]^+ = \rho_{\alpha \beta}
$

and thus we have to solve

$
A_\alpha = \sum_\beta \rho_{\alpha\beta}
$

where the $A_\alpha$ are determined by $-\mu^{-1}\text{div}_\alpha(\mu \Phi_\alpha)$. A simple solution is given by

$
\rho_{\alpha \beta} = (A^\alpha-A^\beta)/n
$

where $n$ is the number of split states. This solution gives us a direct interpretation of the rates: it is possible to transition into a state $\beta$ _precisely_ when the divergence of flow associated to $\beta$ exceeds the divergence of the flow in the current state $\alpha$. The greater the difference in divergence, the more likely we are to transition into the given state.


Note that the $A_\alpha$ are arbitrary (but must satisfy $\sum_\alpha A_\alpha=0$), and so this is a valid solution and rates for any split. If for some reason we want to have a substantially increased rate we may adjust the rate matrix $\lambda_{\alpha\beta} \to \lambda_{\alpha\beta} + R_{\alpha\beta}$ where $R_{\alpha\beta}$ is any positive definite symmetric matrix. 

As we shall soon see there are several other solutions of interest.

### Preferential splitting rates
Suppose we are interested in a particular state, say, $\alpha = 0$, for which we want to have as small rates $\lambda_{0\beta}$ as possible. In other words, we would like 'stay' in the state $\alpha = 0$ for longer (or, at the very least, have fewer events $0 \to \beta$). Then, as we shall see, the above rates are not ideal. The rate $\lambda_{0 \to \text{any}} = \sum_\beta \lambda_{0\beta}$
becomes

$\lambda_{0 \to \text{any}} = \frac{1}{n+1}\sum_\beta [A_0-A_\beta]^+\geq \frac{1}{n+1}[\sum_\beta A_0-A_\beta]^+ = \frac{1}{n+1}[\sum_\beta A_0]^+ = [A_0]^+$

with equality if and only if $A_0 - A_\beta \geq 0$ for every $\beta$.

Had we only split into $0$ and taken the other splits $\beta= 1,2,\ldots, n $ as a single state $I$ the above 'recipe' would instead declare

$\lambda_{0 \to I} = \frac{1}{2}[A_0 - A_I]^+ =[A_0]^+$

since $A_0 + A_I = 0$. Clearly the former choice means that we transition into $I$ with a frequency that is *at least* as big as that of the latter choice, but generally larger.

Thus, if possible, we would like to use another rate for transitions $0 \to \beta$. Mercifully there are some obvious choices. Considering the equation

$A_0 = \sum_\beta \lambda_{0\beta}-\lambda_{\beta 0 }$

we can, as above, pick an anti-symmetric $\rho$-matrix and $\lambda_{0 \beta} = [\rho_{0\beta}]^+$ and $\lambda_{0 \beta} = [-\rho_{0\beta}]^+$  which leads to

$A_0 = \sum_\beta \rho_{0\beta}$

which we can solve by letting $\rho_{0\beta} = \frac{1}{n} A_0 = -\rho_{\beta 0}$ for $\beta \neq 0$. How should we interpret this choice for the rates? Now we only shift from $\alpha =0$ to _some_ (note that we effectively pick which uniformly at random) $\beta$ if the average divergence in $-A_\beta = A_0$ exceeds $0$, rather than if some individual divergence does so.

This has some consequence for the divergence equations for other splits. Let lower-case latin letters $a \neq 0$ index the $n$ splits other than $\alpha = 0$. We can in fact apply the same recipe as above again. To see this we have

$A_a = \sum_\alpha \lambda_{a\alpha}-\lambda_{\alpha a} =  (\lambda_{a0}-\lambda_{0a}) + \sum_b\lambda_{ab}-\lambda_{b a}  = \rho_{a0} + \sum_b\lambda_{ab}-\lambda_{b a} 
=-\frac{A_0}{n} + \sum_b\rho_{ab}$

Thus, picking $\rho_{ab} = \frac{1}{n}(A_a - A_b)$ we get

$ \sum_b\rho_{ab} = A_a - \frac{1}{n}\sum_b A_b = A_a  - \frac{1}{n}(-A_0 + \sum_{\beta} A_\beta ) = A_a + \frac{1}{n}A_0$

since $\sum_b A_b = -A_0 + \sum_\beta A_\beta = -A_0$ due to the vanishing overall divergence. Hence the $A_0$ terms in $a$-divergence equation cancel and the solution

$\rho_{ab} = \frac{1}{n}(A_a - A_b)$ 

is satisfactory.

### The Lagrangian split-velocity case
We now turn our attention to a splitting of the Lagrangian dynamics, so that we have one state corresponding to evolution of position, and one state for each evolution of a velocity component $v^i$. Thus, in sampling in $\mathbb{R}^n$ we shall have $n+1$ split states. In our sampling we shall treat position updates preferentially (
Shifting between velocities is *hopefully* inexpensive, but shifting between position and velocity updates is quite costly as it incurs a computation of third order derivatives.).

#### Notation, notation, my kingdom for better notation
We let the position flow correspond to the split state variable $\alpha = n+1$, so that the $I$:th velocity component can be chosen to be represented by $\alpha = I$, and so that matrix-enumerations match this (Matrices and arrays in Julia are, of course - as with any other sane language - indexed starting from 1.). We shall, in what comes, adopt a somewhat specialized notation. Namely, we let capital latin latters (typically $I,J,K,\ldots$) denote indices over a single dimension (the splitting dimension) which means we _**do not apply the Einstein summation convention**_ to these indices. By lower-case latin letters we refer to components that run across 'all dimensions other than the one currently evolving'. That is to say, if $e.g. $\alpha = 3$ then a lower case letter $a$ tracks across all indices $1, 2, 4, 5, \ldots, n$. Greek letters refer always to the full set of indices $1,2,3,4,\ldots, n$. So if $\alpha = I$ we have

$\Gamma^\alpha_{\alpha \beta} = \Gamma^I_{I\beta} + \Gamma^{a}_{a\beta}$

but sometimes we encounter expressions like 

$\Gamma^\alpha_{J\beta}v^Jv^\beta = \Gamma^\alpha_{JI}v^Jv^I  + \Gamma^\alpha_{Ja}v^Jv^a$

where for $J\neq I$ the $a$ now does run over $J$, since $I$ is assumed to be evolving. Very tricky.

#### The equation of motion and divergences
The Lagrangian dynamics are described at length in the paper. Let us summarize some key features:

The flow in position, now associated to the $n+1$:th split state, is, componentwise:

$(\Phi_{n+1})^a = v^a$ 

and for the flow in the $I$:the velocity component we have (in a very slight abuse of notation we take $\Phi_I^I = \Phi^I$)

$\Phi^I = (-\eta - G^{-1}\nabla \phi)^I = -\Gamma^I_{\alpha\beta}v^\alpha v^\beta - G^{I\alpha}\partial_\alpha \phi = $

$= -\Gamma^I_{II} (v^I)^2 - 2\Gamma^I_{Ia}v^Iv^a-\Gamma^I_{ab}v^av^b-G^{I\alpha}\partial_\alpha \phi $

and since we do not (primarily) deal with BPS-type events we set $\phi = -\log \pi + \frac{1}{2}\log \det G$. However, since this is independent of $v$ it does not change the qualitative nature of the ODE defined by the flow along $\Phi_{I}$. We can express the flow in $v^J$ as a form of the _Ricatti equation_:

$du/dt = au^2 + b u + c $

for coefficients depending on $J$:

$ a_{;J} = -\Gamma^J_{JJ}$

$ b_{;J} = - 2\Gamma^J_{Ja}v^a$

$ c_{;J} = -G^{J\alpha}\partial_\alpha \phi-\Gamma^J_{ab}v^av^b$

The divergences for the velocities are straightforward to compute, but less pleasant to expand in our specialized indices. We shall need an expression for each divergence $A_I$ as a function of the evolving velocity component $u = v^J$. The divergences are cubic in the velocities so we shall expand $A_{I;J}(u) = A_{I;J}^{(0)} + A_{I;J}^{(1)}u^1+A_{I;J}^{(2)}u^2+A_{I;J}^{(3)}u^3 $ 

$A_I = -\mu^{-1}\text{div}_I(\mu \Phi_I) = (2\Gamma^I_{I\alpha} + \Phi^IG_{I\alpha})v^\alpha = 
2\Gamma^I_{I\alpha}v^\alpha - \Gamma^I_{\mu\nu}G_{I\alpha}v^\mu v^\nu v^\alpha - G^{I\beta} G_{I\alpha}(\partial_\beta \phi)v^\alpha $

and hence

$A_{I;J}^{(0)} = 2\Gamma^I_{Ia}v^a - \Gamma^I_{ab}G_{Ic}v^a v^b v^c - G^{I\beta} G_{Ia}(\partial_\beta \phi)v^a$

$A_{I;J}^{(1)} = 2\Gamma^I_{IJ} - 2\Gamma^I_{aJ}G_{Ib}v^a  v^b - \Gamma^I_{ab}G_{IJ}v^a v^b  - G^{I\beta} G_{IJ}(\partial_\beta \phi) $

$A_{I;J}^{(2)} = -\Gamma^I_{JJ} G_{Ia}v^a - 2\Gamma^I_{Ja}G_{IJ}v^a  $

$A_{I;J}^{(3)} = -\Gamma^I_{JJ}G_{IJ}$

The final divergence is

$A_{n+1; J} = -2\Gamma^\alpha_{\alpha \beta}v^\beta + \Gamma^\mu_{\alpha \beta} G_{\mu \nu} v^\alpha v^\beta v^\nu + (\partial_\alpha \phi)v^\alpha$

so (as might be inferred also from the above expression, and knowing that $A_{n+1} = \mu^{-1}\text{div}_x(\mu \Phi) = -\mu^{-1}\text{div}_v(\mu \Phi) = \sum_a A_a$)

$A_{n+1; J}^{(0)}  = -2\Gamma^\alpha_{\alpha a}v^a + \Gamma^\mu_{ab} G_{\mu c} v^a v^b v^c + (\partial_a \phi)v^a$

$A_{n+1; J}^{(1)}  = -2\Gamma^\alpha_{\alpha J} + 2\Gamma^\mu_{J a} G_{\mu b} v^a v^b + \Gamma^\mu_{ab } G_{\mu J} v^a v^b + (\partial_J \phi)$

$A_{n+1; J}^{(2)}  =  2\Gamma^\mu_{J a} G_{\mu J} v^a + \Gamma^\mu_{JJ}G_{\mu a}$

$A_{n+1; J}^{(3)}  =  \Gamma^\mu_{J J} G_{\mu J} $

Of course, to compute $A_{n+1}$ we should certainly use the form of the $A_a$ above. Since, at all times, we are only interested in a single $J$ at a time, and the relationships between $A_{I;J}$ and $A_{I;K}$ are somewhat complicated, we do not benefit greatly from computing the full $A_{I;J}^{(n)}$.

## Implementing the flow, divergences, rates etc

In [1]:
#Computing the object A^{(n)}_{I;J}:
function compute_divergences!(A, reduced_v, dim, J,  Γ, G, G_inv, ∇φ, vel)
    reduced_v .= vel
    reduced_v[J] = 0.0
    
    @views A[dim+1, :] .= 0.0

    @inbounds for I in 1:dim
        @views A[I, 4] = -Γ[I, J, J] * G[I, J]
        
        A[dim+1, 4] -= A[I, 4]

        @views A[I, 3] = -Γ[I, J, J] * dot(G[I, :], reduced_v) - 2 * G[I, J] * dot(Γ[I, J, :], reduced_v)

        A[dim+1, 3] -= A[I, 3]

        @views A[I, 2] = 2*Γ[I, I, J] - 2*dot(Γ[I, J, :], reduced_v)*dot(G[I, :], reduced_v) - G[I, J] * dot(reduced_v, Γ[I, :, :], reduced_v) - G[I, J] * dot(G_inv[I, :], ∇φ)

        A[dim+1, 2] -= A[I, 2]

        @views A[I, 1] = 2*dot(Γ[I, I, :], reduced_v) - dot(reduced_v, Γ[I, :, :], reduced_v)*dot(G[I, :], reduced_v) - dot(G[I, :], reduced_v) * dot(G_inv[I, :], ∇φ)

        A[dim+1, 1] -= A[I, 1]
    end

    return A
end        

compute_divergences! (generic function with 1 method)

As a rough order of magnitude estimate this should take something like $\sim 10 \mu s$ to compute for a 20-dimensional space.

#### $J\to I$ and $J\to 0$ rate under $J$-flow from the divergences
We have $\rho_{JI} = (A_{J;J}-A_{I;J})/n $ and $\rho_{J , n+1} = -A_{n+1;J}/n$ so given the above we can compute the rates with ease.

In [2]:
function compute_rho!(ρJ, J, A, dim)
    @inbounds for I in 1:dim
        if I ≠ J
            @views ρJ[I,:] .= A[J,:] - A[I,:]
        else
            @views ρJ[J,:] .= 0.0
        end
    end

    @views ρJ[n+1, :] .= (-A[n+1,:] ./ dim) 

    return ρJ
end

compute_rho! (generic function with 1 method)

### Ricatti equation and the velocity flow
The fully split velocity satisfies the Ricatti equation

$du/dt = a u^2 + b u + c$

for some $a,b,c$. This admits the solution

$u(t) = (\kappa \tan(\kappa(t+t_0))-(b/2))/a$

where $\kappa = \sqrt{4ac-b^2}/2$

Notably, if $\kappa$ is imaginary $\kappa = i k$ for some $k \in \mathbb{R}$ then

$\kappa \tan(\kappa s) = i k \tan( iks ) = -k\text{tanh}(ks)$

In [4]:
riccati(t, a, b, c; t0 = 0.0) = a*c-(b/2)^2 < 0 ? ricatti_κ(t, a, b, sqrt(abs(a*c-(b/2)^2)), t0 = t0, imag = true) : ricatti_κ(t, a, b, sqrt((a*c-(b/2)^2)), t0 = t0, imag = false)
ricatti_κ(t, a, b, κ; t0 = 0.0, imag::Bool = false) = imag ? (κ * tan(κ*(t+t0))-(b/2))/a : (-κ * tanh(κ*(t+t0))-(b/2))/a

@benchmark ricatti_κ($0.2, $1.0, $2.0, $3.0)

BenchmarkTools.Trial: 10000 samples with 998 evaluations per sample.
 Range (min … max):  19.940 ns … 163.327 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     21.142 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   22.656 ns ±   5.934 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▃█▇▁ ▁   ▅▄         ▁▂                                       ▁
  ████▄██▇▇███▅▄▄▄▄▆▅▇██▇▆▅▆▅▅▆▇▅▄▄▅▆▅▅▄▅▆▇▇██▇▆▅▄▄▄▃▄▄▃▃▄▃▄▃▅ █
  19.9 ns       Histogram: log(frequency) by time      47.5 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

## Exact methods for rate integrals
### Exact methods
The rates are given by some expression

$\lambda^{IJ} = [\rho^{IJ}]^+ = [A^I - A^J]^+/n$

and the total rate in state $I$ is simply the sum $\lambda^I = \sum_J\lambda^{IJ}$. As discussed above the $A^I$ are cubic in $u(t)$. We can explicitly integrate each $\lambda^{IJ}$ *if* we know that it is positive. Thus, for a given $I$, we solve $\rho^{IJ}(u) = 0$ for all $J$ and order the individual solutions $u_1, u_2, \ldots$ such that $u_i < u_{i+1}$ if $du/dt > 0$, and $u_i> u_{i+1}$ else (Note that $du/dt$ is actually identically positive or negative for all $t$ we will consider, since $u(t)$ 'blows up' in finite time (this is a weird argument - it is true because we know that $u(t)$ has to blow up in finite time, and has to be a solution of the Ricatti equation)). This partitions the velocity space $U$ into sets over which the signs of all $\rho^{IJ}$ are constant and over these the sum $\sum_J \lambda^{IJ}$ can thus be computed.

In [5]:
pol = [9,3,-7,1]
@benchmark roots($pol)

BenchmarkTools.Trial: 10000 samples with 193 evaluations per sample.
 Range (min … max):  488.601 ns … 191.056 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     530.052 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   739.673 ns ±   2.742 μs  ┊ GC (mean ± σ):  7.05% ± 5.36%

  ▆█▅▅▃▄▃▂▁▁▁▁▁▂▂▂                                              ▁
  ███████████████████████▇▇▇▆▆▆▆▆▆▆▅▄▃▅▅▄▄▃▄▃▃▅▄▃▄▄▃▃▄▃▃▄▂▃▄▃▂▄ █
  489 ns        Histogram: log(frequency) by time       2.36 μs <

 Memory estimate: 496 bytes, allocs estimate: 8.

In [6]:
"""
    real_roots(b, c, d; verbose=false)::Tuple

Finds the real roots of the cubic polynomial x^3 + b x^2 +c x + d.
"""
function real_roots(b, c, d; verbose=false)::Tuple
    Δ = b^2 - 3*c
    μ = (2*(b^3)) - (9*b*c) + (27*d)

    if iszero(Δ) && iszero(μ) #Should check vs threshold.
        return (-b/3,) 
    end

    #Optimize: Add special statement for when Δ = 0 (within threshold.)
    L = (μ^2)-(4*Δ^3)
    if L < 0 
        verbose ? println("L < 0") : nothing
        z = (μ + sqrt(-L)*im) #2z = ...,  but we only use the angle for z
        r2 = cbrt((μ^2 - L)/4)
        if r2 ≈ Δ #This must be handled more carefully!
            verbose ? println("r2 = Δ, difference: $(r2-Δ)") : nothing

            (s, c) = sincos(angle(z)/3) #can be optimized/altered to use the 'tan-formula' for the 3xReal root cubic
            k = 2*sqrt(r2)
            return (b .+ (k.* (c, (-c + (sqrt(3)*s))/2, (-c - (sqrt(3)*s))/2))) ./(-3)
        else
            verbose ? println("r2 ≠ Δ, difference: $(r2-Δ)") : nothing
            if μ ≤ 0
                verbose ? println("μ ≤ 0") : nothing
                return (b + sqrt(r2)*(1+(Δ/(r2))),) ./(-3)
            else
                verbose ? println("μ > 0") : nothing
                return (b - sqrt(r2)*(1+(Δ/(r2))),)./(-3)
            end
        end
    elseif L ≥ 0
        verbose ? println("L ≥ 0, L: $L") : nothing
        root = sqrt(L)
        if μ > 0
            z = (μ + root)/2
        else
            z = (μ - root)/2
        end
            
        C = cbrt(z)
        r2 = C^2
        if r2 ≈ Δ #This must be handled more carefully! We should check roots (amounts to a single computation) 
            verbose ? println("r2 = Δ, difference: $(r2-Δ)") : nothing
            return (b+2*C, b-C) ./ (-3)
        else
            verbose ? println("r2 ≠ Δ, difference: $(r2-Δ)") : nothing
            return (b + C*(1+(Δ/r2)),) ./(-3)
        end
    end
end

real_roots(a,b,c,d) = real_roots(b/a, c/a, d/a)

real_roots (generic function with 2 methods)

Thus we have established a pretty decent root-finder. Let's look at its performance:

In [7]:
bm_roots_1R = @benchmark real_roots($2.0, $(-3.0), $9.0) #One real root

BenchmarkTools.Trial: 10000 samples with 984 evaluations per sample.
 Range (min … max):  56.199 ns … 961.077 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     59.451 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   67.556 ns ±  25.619 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▄▇█▂▁▁▁▃▃     ▁  ▂▅▃▃▁                                       ▁
  ████████████▇██▇▇██████████▇▇▇▇▇█▆▆▅▆▅▅▅▄▅▄▄▂▄▄▄▂▅▅▅▃▄▄▅▄▄▂▄ █
  56.2 ns       Histogram: log(frequency) by time       142 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [ ]:
bm_roots_3R = @benchmark real_roots($(-7.0), $(3.0), $9.0) #Three real roots

This is something like $\sim 10-15$ times faster than the one defined above. As a rough measure of performance then we expect something like $\sim 80 N ns$ to compute the partition of the time interval for a velocity update. For e.g. the 20-dimensional Twin Peaks scenario $N = 21$ so we get something akin to $\sim 1.6 \mu s$ of work. Of course, if we wanted to we could parallellize this particular task, but the overhead for such an endeavour would probably be prohibitive. 

#### Partitioning the time intervals

In [ ]:
real_roots(X::SVector{4, Float64}) = real_roots(X[1], X[2], X[3], X[4])
real_roots(X::AbstractArray) = real_roots(X[1], X[2], X[3], X[4])

In [ ]:
A1=@SVector rand(4)
@benchmark real_roots($A1) 

In [ ]:
function partition_by_roots!(T::BinaryMinHeap{Float64}, I::Integer, ρ::Array{Float64, 3}, a, b, κ; start_time = 0.0) #could be $MVector
    empty!(T)
    #push!(T, start_time)
    if isreal(κ)
        t(u) = atan((a*u + (b/2))/κ) - start_time 
    else
        k = imag(κ)
        t(u) = atanh((a*u + (b/2))/k) - start_time
    else 
    ρ_I = @view ρ[I,:,:]
    for J in axes(ρ_I, 1)
        roots = real_roots(@view(ρ_I[J,:]))
        for root in roots
            #The root corresponds to a velocity for which the rate becomes positive.
            #We transform it into a time.
            root_time = t(root)  
            if root_time > start_time
                push!(T, root_time)
            end
        end
    end
    return T
end            

In [ ]:
D = 21
ρ = rand(21, 21, 4);
T = BinaryMinHeap{Float64}()
a = rand()
b = rand()
κ = rand()
T = partition_by_roots!(T, 1, ρ, a, b, κ)

In [ ]:
@benchmark partition_by_roots!($T, $1, $ρ, $a, $b, $κ)

We get roughly to $2\mu s$ of computation time, indicating that the transformation of each root takes a slight amount of time.

#### The rate integrals 
The rates need to be integrated, that is we integrate

$\lambda = A u^3 + B u^2 + C u + D = \bar{A} \cdot \bar{U}$

over time (on the intervals discussed above). Each integral

$M_n = \int u(t)^n dt$

can be analytically computed. We expand in the $\tan(\kappa (t+t_0))$ after a transformation $s = \kappa(t+t_0)$ so that

$dt = ds/\kappa$

and (by abuse of notation)

$u(s) = (\kappa \tan(s) -(b/2))/a$

whence

$M_n(s) = \frac{1}{a^n\kappa} \sum_{j=0}^n \binom{n}{j} (-b/2)^{n-j} \int (\kappa \tan(s))^{j}ds = \frac{1}{a^n\kappa} \sum_{j=0}^n \binom{n}{j} \kappa^{j}(-b/2)^{n-j} L_{j} = \frac{1}{a^n\kappa}\bar{M_n} \cdot \bar{L}$

with each $L_j$ corresponding to a $\tan(x)^j$ integral. It can be shown that

1) $L_0 = s$
2) $L_1 = -\log(\cos(s))$
3) $L_2 = (\tan(s)-s)$
3) $L_3 = (\frac{1}{2\cos^2(s)} + \log(\cos(s)))$

Obviously there are some issues with the values - they can be divergent, complex and otherwise problematic. However, we shall always have $s$ in specific domains, and the integrals shall only be integrated over domains for which $\lambda \geq 0$.

Of course we shall typically evaluate this at many distinct "times" $s$, which alters the vector $L$. Thus it makes sense to use a matrix $M$ with the distinct $M_i$ as rows so that $\bar{U}(s) = M \bar{L}(s)$

In [ ]:
function L_tuple(s::Float64)
    c = cos(s)
    lc = log(c)
    return (s, -lc, tan(s) -s , lc + (1. /(2*(c^2))))
end

In [ ]:
function M_matrix!(M::MMatrix{4,4, Float64, 16}, a, b, κ)
    for i in 1:4
        for j in 1:i
            M[i, j] = binomial(i, j) * ((-b / 2.)^j) * (κ^(i-j))
        end
        factor = (κ * (a^i))
        @views M[i,:] ./= factor
    end
    return M 
end

In [ ]:
M = @MMatrix zeros(4,4)
M_matrix!(M, rand(), randn(), rand())

In [ ]:
@benchmark M_matrix!($M, $a, $b, $κ)

In [ ]:
function compute_rate_integrals(ρ_IJs::Vector{SVector{4, Float64}}, M::MMatrix, initial_values::Vector{Float64}, s_final::Float64)
    